# ■ 딥러닝 컴페티션

### 컴페티션의 목표는 ① 학습 평가의 이해, ② 딥러닝 성능 개선 방법 숙지, ③ 설명력을 키우는게 목적

## ■ 공지

※ 모델 성능이 제일 높은 기준으로 채점 X (상대평가는 안하지만, 개인 성능은 valid score가 75이상 나오길 권장함.)

※ 수업 코드 사용 가능

※ 학습자 간 상의 절대 금지.

※ 최신 기술 또는 대단한 아이디어 보다. ① 코딩의 인과성, ② 설명의 깊이만 충분하다면, 만점이 가능합니다.

</br>

## ■ 평가기준 (총점 90점) ※ 글자 수로 점수 평가 X

1. (15점) 전처리 아이디어 적합성 + 논리 (650자 이내, 주석 서술하기)

2. (20점) EDA를 통한 타당한 해석 (650자 이내, 주석 서술하기)

3. (25점) Feature Selection과 모델 선택, 튜닝 기준 (650자 이내, 주석 서술하기)

4. (25점) 개선사항 (650자 이내, 주석 서술하기)

5. (5점 ) validation score 적절하게 출력 (300자 이내, 주석 서술하기)

</br>

## ■ 깃허브 정리 (10점)

1. 포트폴리오로 쓸 수 있도록 프로젝트 제목, 전처리, EDA, 모델링 방법, 성능 결과를 캡처 이미지와 함께 잘 정리.

</br>

## ■ 제출방법

5월 12일 23시 59분까지, 오승환 강사에게 DM으로 ipynb 파일 제출, 이후 깃허브 링크도 정리되는대로 DM 제출

# 1. 원본 데이터 출처

https://www.kaggle.com/datasets/parisrohan/credit-score-classification

# 2. 데이터 클리닝 방법 출처:

https://www.kaggle.com/code/clkmuhammed/credit-score-classification-part-1-data-cleaning#Download-Link

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

file_path = 'train2.csv'
data = pd.read_csv(file_path)

data = data.drop(columns=['ID', 'Customer_ID', 'Name', 'SSN'])

categorical_columns = ['Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

target_encoder = LabelEncoder()
data['Credit_Score'] = target_encoder.fit_transform(data['Credit_Score'])

X = data.drop('Credit_Score', axis=1).values
y = data['Credit_Score'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class CreditScoreDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = CreditScoreDataset(X_train, y_train)
test_dataset = CreditScoreDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_size, 64)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.layer3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


input_size = X_train.shape[1]
model = MLP(input_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total_train += targets.size(0)
        correct_train += (predicted == targets).sum().item()

    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total_val += targets.size(0)
            correct_val += (predicted == targets).sum().item()

    train_accuracy = 100 * correct_train / total_train
    val_accuracy = 100 * correct_val / total_val
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, '
          f'학습 정확도: {train_accuracy:.2f}%, 평가 정확도: {val_accuracy:.2f}%')


Epoch [1/20], Loss: 0.6881, 학습 정확도: 68.50%, 평가 정확도: 69.90%
Epoch [2/20], Loss: 0.6517, 학습 정확도: 70.28%, 평가 정확도: 70.29%
Epoch [3/20], Loss: 0.6416, 학습 정확도: 70.70%, 평가 정확도: 70.51%
Epoch [4/20], Loss: 0.6355, 학습 정확도: 70.95%, 평가 정확도: 69.99%
Epoch [5/20], Loss: 0.6309, 학습 정확도: 71.13%, 평가 정확도: 70.95%
Epoch [6/20], Loss: 0.6266, 학습 정확도: 71.44%, 평가 정확도: 70.44%
Epoch [7/20], Loss: 0.6234, 학습 정확도: 71.49%, 평가 정확도: 70.87%
Epoch [8/20], Loss: 0.6203, 학습 정확도: 71.62%, 평가 정확도: 70.70%
Epoch [9/20], Loss: 0.6167, 학습 정확도: 71.63%, 평가 정확도: 70.72%
Epoch [10/20], Loss: 0.6152, 학습 정확도: 71.78%, 평가 정확도: 70.85%
Epoch [11/20], Loss: 0.6119, 학습 정확도: 72.04%, 평가 정확도: 71.11%
Epoch [12/20], Loss: 0.6093, 학습 정확도: 72.17%, 평가 정확도: 71.11%
Epoch [13/20], Loss: 0.6071, 학습 정확도: 72.26%, 평가 정확도: 71.39%
Epoch [14/20], Loss: 0.6052, 학습 정확도: 72.36%, 평가 정확도: 70.98%
Epoch [15/20], Loss: 0.6030, 학습 정확도: 72.38%, 평가 정확도: 71.48%
Epoch [16/20], Loss: 0.6002, 학습 정확도: 72.59%, 평가 정확도: 71.44%
Epoch [17/20], Loss: 0.5980, 학습 정확도: 72.66%, 평가 정

In [2]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import os
import warnings
warnings.filterwarnings('ignore')



# 하이퍼파라미터 설정

# 학습 기본 설정
EPOCHS        = 100      # 학습 반복 횟수 (Early Stopping으로 자동 종료)
#                         └ 과적합 없이 충분히 학습하려면 100 이상 권장
BATCH_SIZE    = 256      # 한 번에 학습할 데이터 수
#                         └ GPU 메모리 여유 있으면 512로 올리기 / 작을수록 일반화 좋지만 느림
LEARNING_RATE = 5e-4     # 학습률
#                         └ 성능 안 오를 때: 1e-4로 낮추기
#                         └ 학습이 너무 느릴 때: 1e-3으로 올리기
WEIGHT_DECAY  = 1e-4     # 과적합 방지 L2 정규화 강도
#                         └ 과적합 심하면 1e-3으로 올리기
DROPOUT_RATE  = 0.3      # Dropout 비율
#                         └ 과적합 심하면 0.5까지 올리기 / 학습 안되면 0.1로 낮추기
SEED          = 42       # 랜덤 시드 (재현성)

# 모델 구조 설정
HIDDEN_LAYERS = [512, 256, 128, 64]  # 은닉층 크기
#                         └ 데이터 많으면 [1024, 512, 256, 128, 64]로 확장
#                         └ 과적합이면 [256, 128, 64]로 축소
ACTIVATION    = 'GELU'   # 활성화 함수: 'ReLU' / 'LeakyReLU' / 'GELU'
#                         └ GELU가 일반적으로 MLP에서 가장 좋은 성능
USE_BATCHNORM = True     # BatchNorm 사용 여부
USE_RESIDUAL  = True     # Residual Connection 사용 여부 (입력 → 마지막 은닉층으로 skip)
#                         └ 층이 깊을수록 True 권장 (기울기 소실 방지)

# 학습 전략 설정
K_FOLDS       = 5        # Cross Validation fold 수
PATIENCE      = 15       # Early Stopping: 개선 없으면 몇 epoch 후 종료
SCHEDULER     = 'cosine' # LR 스케줄러: 'cosine' / 'step' / 'none'

# 앙상블 설정
USE_ENSEMBLE  = True     # XGBoost + LightGBM 앙상블 여부
ENSEMBLE_MODE = 'soft'   # 'soft' (확률 평균) / 'hard' (다수결)
ENSEMBLE_WEIGHTS = {'mlp': 0.25, 'xgb': 0.45, 'lgb': 0.30}
#                         └ 트리 모델이 본 데이터에서 더 강해 가중치를 높게 줌

# 랜덤 시드 고정
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'하이퍼파라미터 설정 완료 | Device: {device}')

# 시각화 결과 저장 폴더
os.makedirs('images', exist_ok=True)
sns.set_style('whitegrid')



# 데이터 로딩 및 전처리

# ① 식별자 컬럼 처리: ID/Name/SSN은 그대로 제거하되, Customer_ID는 *제거 직전에*
#    groupby에 활용한다. 이 데이터는 한 고객당 8개월치 행이 들어있는 패널 구조
#    (고유 고객 12,500명 × 8개월 = 10만 행)라서, 같은 고객의 과거 통계량이 단일
#    시점 값보다 신용도를 훨씬 잘 설명한다. 집계 후엔 Customer_ID도 제거.

# ② Type_of_Loan 처리: 원본은 "auto loan, student loan, ..." 형식의 다중값 문자열
#    (unique 6,261개). 기본 코드처럼 LabelEncoder로 묶으면 의미 없는 문자열 조합에
#    임의의 순서를 부여하게 된다. 8개 주요 대출 유형 각각의 보유 여부를 0/1로
#    분리하는 Multi-hot Encoding으로 정보 손실을 막았다.

# ③ 다중공선성 제거: Annual_Income과 Monthly_Inhand_Salary는 상관계수 0.998로
#    사실상 동일 정보. Monthly_Inhand_Salary를 제거해 입력의 중복을 줄였다.

# ④ 비율 피처 생성: 절대값(부채 1,000달러)보다 비율(연봉 대비 부채 30%)이
#    신용 평가의 본질에 가깝다. 부채/소득, EMI/소득, 투자금/소득 비율 추가.


file_path = 'train2.csv'
data = pd.read_csv(file_path)

# EDA 시각화용 원본 보존 (전처리 전 데이터 분포를 그대로 보기 위함)
raw_data = data.copy()

# Customer_ID 기반 집계 피처 생성 (제거 전에 활용)
agg_cols = ['Annual_Income', 'Outstanding_Debt', 'Num_of_Delayed_Payment',
            'Delay_from_due_date', 'Credit_Utilization_Ratio', 'Monthly_Balance',
            'Num_Credit_Inquiries', 'Changed_Credit_Limit', 'Num_Credit_Card',
            'Num_Bank_Accounts', 'Interest_Rate', 'Total_EMI_per_month',
            'Amount_invested_monthly']

for col in agg_cols:
    data[col + '_mean'] = data.groupby('Customer_ID')[col].transform('mean')
    data[col + '_std']  = data.groupby('Customer_ID')[col].transform('std').fillna(0)

# 비율 피처 (분모 0 방지 위해 +1)
data['Debt_Income_Ratio']   = data['Outstanding_Debt']             / (data['Annual_Income'] + 1)
data['EMI_Income_Ratio']    = data['Total_EMI_per_month'] * 12     / (data['Annual_Income'] + 1)
data['Invest_Income_Ratio'] = data['Amount_invested_monthly'] * 12 / (data['Annual_Income'] + 1)

# 식별자 + 다중공선성 컬럼 제거
data = data.drop(columns=['ID', 'Customer_ID', 'Name', 'SSN', 'Monthly_Inhand_Salary'])

# Type_of_Loan을 Multi-hot Encoding으로 분해
loan_types = ['auto loan', 'credit-builder loan', 'personal loan',
              'home equity loan', 'mortgage loan', 'student loan',
              'debt consolidation loan', 'payday loan']
for lt in loan_types:
    col_name = 'loan_' + lt.replace(' ', '_').replace('-', '_')
    data[col_name] = data['Type_of_Loan'].str.lower().str.contains(lt, na=False).astype(int)
data = data.drop(columns=['Type_of_Loan'])

# 나머지 범주형은 기본 코드와 동일하게 LabelEncoder
categorical_columns = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

# 타겟 인코딩
target_encoder = LabelEncoder()
data['Credit_Score'] = target_encoder.fit_transform(data['Credit_Score'])

X_all = data.drop('Credit_Score', axis=1).values
y_all = data['Credit_Score'].values
feature_names = data.drop('Credit_Score', axis=1).columns.tolist()



# EDA - 시각화 6종

# ① 클래스 분포: Standard 53.2% / Poor 29.0% / Good 17.8%로 불균형이지만
#    극단적이지 않다. class_weight 적용 시 전체 accuracy가 떨어져 미적용.

# ② Mutual Information 상위권: Annual_Income, Amount_invested_monthly,
#    Outstanding_Debt가 1군(MI≈0.6대). Annual_Income과 Monthly_Inhand_Salary가
#    둘 다 상위 → 다중공선성 시사 → 히트맵에서 0.998 확인.

# ③ 상관 히트맵: 부채·연체 관련 변수들(Outstanding_Debt, Num_of_Delayed_Payment,
#    Interest_Rate)이 0.5~0.6대로 강하게 묶임 → "재정 부담군" 클러스터 형성.

# ④ 박스플롯: 연체 횟수·일수는 Good→Poor로 갈수록 명확히 증가. 반면
#    Annual_Income은 분포 겹침이 큼 → 비율 피처(부채/소득)가 필요한 신호.

# ⑤ 대출 유형별 신용 분포: 사전 추측과 달리 8개 대출 유형 모두 신용 등급 분포가
#    거의 동일(Good≈12%, Standard≈50%, Poor≈38%). 대출 종류보다 *관리 방식*이
#    중요. Multi-hot 컬럼은 상호작용 학습에 기여 가능해 유지.

# ⑥ 패널 구조: 모든 고객이 8개월씩 등장. 같은 고객도 시점에 따라 등급 변동
#    (1개로 일관 5,208명, 2개 섞임 7,262명) → 고객별 평균/표준편차가 노이즈를
#    줄여 더 안정적 시그널.

order = ['Good', 'Standard', 'Poor']
colors = ['#2ecc71', '#3498db', '#e74c3c']

# (1) 클래스 분포
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
counts = raw_data['Credit_Score'].value_counts().reindex(order)
ax[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
ax[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 800, f'{v:,}', ha='center', fontweight='bold')
ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
          colors=colors, startangle=90, wedgeprops={'edgecolor': 'black'})
ax[1].set_title('Class Distribution (Ratio)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/01_class_distribution.png', dpi=120, bbox_inches='tight')
plt.close()

# (2) Mutual Information - 라이트 전처리만 적용한 원본 컬럼 기준
mi_data = raw_data.drop(columns=['ID', 'Customer_ID', 'Name', 'SSN']).copy()
for col in ['Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']:
    mi_data[col] = LabelEncoder().fit_transform(mi_data[col].astype(str))
y_mi = LabelEncoder().fit_transform(mi_data['Credit_Score'])
X_mi = mi_data.drop('Credit_Score', axis=1)
mi = mutual_info_classif(X_mi, y_mi, random_state=SEED)
mi_df = pd.DataFrame({'feature': X_mi.columns, 'MI': mi}).sort_values('MI', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mi_df['feature'], mi_df['MI'], color='#3498db', edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 15 Features by Mutual Information', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/02_mutual_information.png', dpi=120, bbox_inches='tight')
plt.close()

# (3) 상관 히트맵 - 주요 수치형 피처
num_cols = ['Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
            'Num_of_Delayed_Payment', 'Delay_from_due_date',
            'Credit_Utilization_Ratio', 'Total_EMI_per_month',
            'Amount_invested_monthly', 'Monthly_Balance', 'Interest_Rate']
corr = raw_data[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/03_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()

# (4) 주요 피처 vs Credit_Score 박스플롯
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
key_features = ['Annual_Income', 'Outstanding_Debt', 'Num_of_Delayed_Payment', 'Delay_from_due_date']
for i, feat in enumerate(key_features):
    r, c = i // 2, i % 2
    sns.boxplot(data=raw_data, x='Credit_Score', y=feat, order=order,
                palette=colors, ax=axes[r, c], showfliers=False)
    axes[r, c].set_title(f'{feat} by Credit_Score', fontweight='bold')
    axes[r, c].set_xlabel('')
plt.tight_layout()
plt.savefig('images/04_features_vs_target.png', dpi=120, bbox_inches='tight')
plt.close()

# (5) 대출 유형별 신용 등급 분포
loan_score_ratio = []
for lt in loan_types:
    mask = raw_data['Type_of_Loan'].str.lower().str.contains(lt, na=False)
    sub = raw_data.loc[mask, 'Credit_Score'].value_counts(normalize=True).reindex(order, fill_value=0)
    loan_score_ratio.append([lt] + sub.tolist())
ratio_df = pd.DataFrame(loan_score_ratio, columns=['loan_type', 'Good', 'Standard', 'Poor'])

fig, ax = plt.subplots(figsize=(11, 5.5))
x_pos = np.arange(len(ratio_df))
bottom = np.zeros(len(ratio_df))
for cls, color in zip(['Good', 'Standard', 'Poor'], colors):
    ax.bar(x_pos, ratio_df[cls], bottom=bottom, label=cls, color=color, edgecolor='black')
    bottom += ratio_df[cls].values
ax.set_xticks(x_pos)
ax.set_xticklabels(ratio_df['loan_type'], rotation=25, ha='right')
ax.set_ylabel('Ratio')
ax.set_title('Credit_Score Distribution by Loan Type', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('images/05_loan_type_distribution.png', dpi=120, bbox_inches='tight')
plt.close()

# (6) 패널 구조 확인
cust_counts = raw_data.groupby('Customer_ID').size()
cust_consistency = raw_data.groupby('Customer_ID')['Credit_Score'].nunique()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(cust_counts, bins=range(0, 12), color='#9b59b6', edgecolor='black', align='left')
ax[0].set_title('Rows per Customer (Panel Structure)', fontweight='bold')
ax[0].set_xlabel('Number of Records')
ax[0].set_ylabel('Number of Customers')

consistency_counts = cust_consistency.value_counts().sort_index()
ax[1].bar(consistency_counts.index, consistency_counts.values, color='#1abc9c', edgecolor='black')
ax[1].set_title('Unique Credit_Scores per Customer', fontweight='bold')
ax[1].set_xlabel('Unique Classes per Customer')
ax[1].set_ylabel('Number of Customers')
ax[1].set_xticks([1, 2, 3])
for i, v in zip(consistency_counts.index, consistency_counts.values):
    ax[1].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('images/06_customer_panel.png', dpi=120, bbox_inches='tight')
plt.close()

print(f'전처리 후 데이터 shape: {X_all.shape}')
print(f'클래스 분포: {np.bincount(y_all)} (Good, Poor, Standard 순)')
print('EDA 시각화 6종을 images/ 폴더에 저장 완료')



# Hold-out Test 분리 (최종 평가용)

# 전체 데이터를 80:20으로 나눠, 20%는 학습에 절대 노출되지 않는 hold-out test로 둔다.
# 나머지 80%로 K-Fold CV를 수행하며, fold마다 검증 세트와 hold-out test에 동시에
# 예측해서 OOF + 최종 앙상블 두 가지를 모두 얻는다.


X, X_test, y, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
print(f'학습용(K-Fold): {X.shape} | Hold-out Test: {X_test.shape}')



# 4. PyTorch Dataset & 모델 정의

# 모델 후보별 검토:
# - TabTransformer: 범주형 어텐션. LabelEncoding 후 거의 수치형이라 어텐션 이점 미미.
# - TabNet: Sequential Attention 마스킹. 구현 복잡도 대비 MLP와 큰 차이 없음.
# - MLP + Residual (채택): 기본 코드의 layer1-layer2-layer3 구조를 유지하면서,
#   깊이를 늘리고 입력→마지막 은닉층 skip connection을 추가. 깊은 MLP의 기울기 소실
#   완화 + 입력의 raw 정보를 마지막까지 보존.
# - XGBoost / LightGBM: 정형 데이터에서 강한 트리 기반 모델. 본 데이터에서 단독으로
#   83% 이상 달성 → 앙상블에서 핵심 역할.

# Feature Selection:
# - MI 하위 피처도 모두 유지. 실험에서 하위 10개를 제거하면 오히려 0.5%p 하락.
#   신경망/부스팅 모두 자동으로 무관 피처 가중치를 낮춰 무시 → 제거 이득보다 손실이 큼.

# 튜닝 기준:
# - AdamW(weight_decay): L2 정규화 내재화
# - CosineAnnealing: lr을 코사인으로 감소 → 초반 탐색 + 후반 미세조정
# - Gradient Clipping: BatchNorm + 큰 lr 조합의 폭발 방지
# - Early Stopping(patience=15): val_acc 정체 시 자동 종료 → 과적합/시간 낭비 방지
# - K-Fold(5): 단일 split의 운에 좌우되지 않는 안정적 성능 평가, fold별 예측 평균으로
#   앙상블 강화
# - 가중 앙상블: 트리 모델이 더 강력하므로 XGB/LGB에 높은 가중치 부여


class CreditScoreDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def get_activation(name):
    """ACTIVATION 설정값에 따라 활성화 함수 반환"""
    return {'ReLU': nn.ReLU(), 'LeakyReLU': nn.LeakyReLU(0.1), 'GELU': nn.GELU()}[name]


class MLP(nn.Module):
    """
    기본 코드의 layer1-layer2-layer3 구조를 유지하면서 다음을 추가:
    - 폭 확장 + 깊이 추가 (HIDDEN_LAYERS 리스트 기반)
    - BatchNorm + Dropout
    - 입력 → 마지막 은닉층 Residual Skip
    """
    def __init__(self, input_size, hidden_layers=HIDDEN_LAYERS,
                 dropout=DROPOUT_RATE, use_bn=USE_BATCHNORM,
                 use_residual=USE_RESIDUAL, activation=ACTIVATION):
        super(MLP, self).__init__()
        self.use_residual = use_residual

        # 은닉층들을 동적으로 구성
        layers = []
        prev = input_size
        for i, h in enumerate(hidden_layers):
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(get_activation(activation))
            # 마지막 은닉층 직전엔 dropout 미적용 (표현 손상 방지)
            if i < len(hidden_layers) - 1:
                layers.append(nn.Dropout(dropout))
            prev = h
        self.hidden = nn.Sequential(*layers)

        # Residual: 입력을 마지막 은닉층 차원으로 투영해서 더함
        if use_residual:
            self.skip = nn.Linear(input_size, hidden_layers[-1])

        # 출력층 (3-class)
        self.output = nn.Linear(hidden_layers[-1], 3)

    def forward(self, x):
        h = self.hidden(x)
        if self.use_residual:
            h = h + self.skip(x)
        return self.output(h)



# K-Fold Cross Validation + 앙상블 학습

# StratifiedKFold로 fold마다 클래스 비율 유지. 각 fold에서 MLP / XGB / LGB를 모두
# 학습하고, hold-out test에 대한 예측 확률을 누적 평균한다.


skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Hold-out test에 대한 모델별 예측 확률 누적용
n_test = len(X_test)
test_proba_mlp = np.zeros((n_test, 3))
test_proba_xgb = np.zeros((n_test, 3))
test_proba_lgb = np.zeros((n_test, 3))

# Fold별 val accuracy 기록
fold_results = {'MLP': [], 'XGB': [], 'LGB': []}

# 학습 곡선 저장용 (가장 높은 val acc를 낸 fold 기준)
best_overall_history = None
best_overall_val = 0.0


def train_mlp_one_fold(X_tr, y_tr, X_va, y_va):
    """한 fold의 MLP를 학습하고 best state, best val acc, history를 반환"""
    model = MLP(X_tr.shape[1]).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    if SCHEDULER == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    elif SCHEDULER == 'step':
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    else:
        scheduler = None

    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(CreditScoreDataset(X_tr, y_tr),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(CreditScoreDataset(X_va, y_va),
                              batch_size=BATCH_SIZE, shuffle=False)

    best_val_acc = 0.0
    best_state = None
    patience_counter = 0
    history = {'loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(EPOCHS):
        # --- 학습 ---
        model.train()
        running_loss = 0.0
        correct_t, total_t = 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total_t += targets.size(0)
            correct_t += (pred == targets).sum().item()

        # --- 검증 ---
        model.eval()
        correct_v, total_v = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                _, pred = torch.max(outputs, 1)
                total_v += targets.size(0)
                correct_v += (pred == targets).sum().item()

        train_acc = 100 * correct_t / total_t
        val_acc   = 100 * correct_v / total_v
        history['loss'].append(running_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Best Checkpoint + Early Stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'    Early stopping at epoch {epoch+1} (best val: {best_val_acc:.2f}%)')
                break

        if scheduler is not None:
            scheduler.step()

    model.load_state_dict(best_state)
    return model, best_val_acc, history


for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f'\n===== Fold {fold}/{K_FOLDS} =====')

    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Scaler는 fold 안에서 train에만 fit
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_test)

    # --- MLP ---
    print(f'  [MLP] 학습 중...')
    mlp_model, mlp_val_acc, history = train_mlp_one_fold(X_tr_s, y_tr, X_va_s, y_va)
    fold_results['MLP'].append(mlp_val_acc)
    print(f'  [MLP] Fold val acc: {mlp_val_acc:.2f}%')

    # MLP의 hold-out test 확률 예측 (소프트맥스로 확률 변환 후 누적 평균)
    mlp_model.eval()
    with torch.no_grad():
        logits = mlp_model(torch.tensor(X_te_s, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    test_proba_mlp += probs / K_FOLDS

    # 가장 잘 학습된 fold의 history 보존 (시각화용)
    if mlp_val_acc > best_overall_val:
        best_overall_val = mlp_val_acc
        best_overall_history = history

    # --- XGBoost ---
    if USE_ENSEMBLE:
        print(f'  [XGB] 학습 중...')
        xgb_model = xgb.XGBClassifier(
            n_estimators=500, max_depth=8, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric='mlogloss', n_jobs=-1, random_state=SEED + fold,
            tree_method='hist', verbosity=0
        )
        xgb_model.fit(X_tr, y_tr)
        xgb_va_acc = accuracy_score(y_va, xgb_model.predict(X_va)) * 100
        fold_results['XGB'].append(xgb_va_acc)
        test_proba_xgb += xgb_model.predict_proba(X_test) / K_FOLDS
        print(f'  [XGB] Fold val acc: {xgb_va_acc:.2f}%')

        # --- LightGBM ---
        print(f'  [LGB] 학습 중...')
        lgb_model = lgb.LGBMClassifier(
            n_estimators=500, max_depth=8, learning_rate=0.05,
            num_leaves=63, subsample=0.85, colsample_bytree=0.85,
            n_jobs=-1, random_state=SEED + fold, verbose=-1
        )
        lgb_model.fit(X_tr, y_tr)
        lgb_va_acc = accuracy_score(y_va, lgb_model.predict(X_va)) * 100
        fold_results['LGB'].append(lgb_va_acc)
        test_proba_lgb += lgb_model.predict_proba(X_test) / K_FOLDS
        print(f'  [LGB] Fold val acc: {lgb_va_acc:.2f}%')


# 학습 곡선 시각화 (best fold)
if best_overall_history is not None:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(best_overall_history['loss'], color='#e74c3c', linewidth=2)
    ax[0].set_title('Training Loss (Best Fold)', fontweight='bold')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')

    ax[1].plot(best_overall_history['train_acc'], label='Train',
               color='#3498db', linewidth=2)
    ax[1].plot(best_overall_history['val_acc'], label='Validation',
               color='#2ecc71', linewidth=2)
    ax[1].axhline(y=75, color='gray', linestyle='--', alpha=0.7, label='Target 75%')
    ax[1].set_title('Accuracy (Best Fold)', fontweight='bold')
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy (%)')
    ax[1].legend()
    plt.tight_layout()
    plt.savefig('images/07_training_curve.png', dpi=120, bbox_inches='tight')
    plt.close()



# 6. 최종 Validation Score (Hold-out Test)

# 전체 데이터를 80:20으로 분할 후, 80%에 대해 5-Fold Stratified CV 수행. fold별로
# MLP + XGB + LGB를 각각 학습해 hold-out test 20%에 대한 확률을 누적 평균(soft
# voting)한다. 가중치는 트리 모델이 더 강한 본 데이터 특성을 반영해
# MLP 0.25 / XGB 0.45 / LGB 0.30 으로 설정.

# 평가는 학습에 한 번도 노출되지 않은 hold-out 20%에서 수행해 일반화 성능을 정직하게
# 측정한다. K-Fold 평균 검증 점수와 최종 hold-out 점수를 함께 출력.


print('\n' + '=' * 60)
print('  K-Fold Cross Validation 평균 검증 점수')
print('=' * 60)
for name, scores in fold_results.items():
    if scores:
        print(f'  {name:3s}: {np.mean(scores):.2f}% (± {np.std(scores):.2f}%)')

print('\n' + '=' * 60)
print('  Hold-out Test 최종 평가')
print('=' * 60)

# 개별 모델 hold-out 성능
mlp_pred = test_proba_mlp.argmax(1)
mlp_test_acc = accuracy_score(y_test, mlp_pred) * 100
print(f'  MLP only:          {mlp_test_acc:.2f}%')

# USE_ENSEMBLE 여부와 관계없이 final_pred를 안전하게 정의
final_pred = mlp_pred
final_acc = mlp_test_acc
ensemble_label = 'MLP only'

if USE_ENSEMBLE:
    xgb_test_acc = accuracy_score(y_test, test_proba_xgb.argmax(1)) * 100
    lgb_test_acc = accuracy_score(y_test, test_proba_lgb.argmax(1)) * 100
    print(f'  XGBoost only:      {xgb_test_acc:.2f}%')
    print(f'  LightGBM only:     {lgb_test_acc:.2f}%')

    # 앙상블 (soft / hard voting)
    if ENSEMBLE_MODE == 'soft':
        w = ENSEMBLE_WEIGHTS
        ensemble_proba = (w['mlp'] * test_proba_mlp +
                          w['xgb'] * test_proba_xgb +
                          w['lgb'] * test_proba_lgb)
        final_pred = ensemble_proba.argmax(1)
    else:  # hard voting
        votes = np.stack([test_proba_mlp.argmax(1),
                          test_proba_xgb.argmax(1),
                          test_proba_lgb.argmax(1)], axis=1)
        final_pred = np.array([np.bincount(row).argmax() for row in votes])

    final_acc = accuracy_score(y_test, final_pred) * 100
    ensemble_label = f'Ensemble ({ENSEMBLE_MODE})'
    print(f'\n  ★ 최종 앙상블 ({ENSEMBLE_MODE}): {final_acc:.2f}%')



# Confusion matrix 시각화
cm = confusion_matrix(y_test, final_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_encoder.classes_,
            yticklabels=target_encoder.classes_, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix ({ensemble_label}, Acc {final_acc:.2f}%)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('images/08_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.close()

print('\n[Classification Report]')
print(classification_report(y_test, final_pred, target_names=target_encoder.classes_))


# 개선사항 정리

# ① XGBoost + LightGBM 앙상블
#    정형 데이터에서 트리 부스팅이 MLP보다 강력. 가중치는 MLP 0.25 / XGB 0.45 / LGB 0.30.

# ② 5-Fold Cross Validation
#    단일 split은 운에 좌우. 5개 fold 예측 평균으로 분산 감소.

# ③ Customer-level 집계 피처
#    주요 13개 컬럼에 고객별 평균·표준편차 추가. "이번 달 부채"보다 "평균 부채와
#    변동성"이 신용도를 훨씬 잘 설명.

# ④ MLP + Residual 구조
#    [512,256,128,64] 확장 + 입력→마지막 은닉층 skip. BatchNorm/Dropout 0.3.

# ⑤ Type_of_Loan Multi-hot
#    6,261개 조합을 8개 대출 유형으로 분리해 0/1 표현.

# ⑥ 학습 전략 : AdamW, CosineAnnealing, Clipping, Early Stop, GELU.
# ⑦ 비율 피처 : 부채/소득, EMI/소득.

# 남은 한계: Customer 집계는 train·val에 같은 고객이 섞여 약한 누수. 신규 고객
# 예측이라면 GroupKFold 필요.


하이퍼파라미터 설정 완료 | Device: cuda
전처리 후 데이터 shape: (100000, 58)
클래스 분포: [17828 28998 53174] (Good, Poor, Standard 순)
EDA 시각화 6종을 images/ 폴더에 저장 완료
학습용(K-Fold): (80000, 58) | Hold-out Test: (20000, 58)

===== Fold 1/5 =====
  [MLP] 학습 중...
  [MLP] Fold val acc: 82.78%
  [XGB] 학습 중...
  [XGB] Fold val acc: 83.14%
  [LGB] 학습 중...
  [LGB] Fold val acc: 82.28%

===== Fold 2/5 =====
  [MLP] 학습 중...
  [MLP] Fold val acc: 82.08%
  [XGB] 학습 중...
  [XGB] Fold val acc: 82.54%
  [LGB] 학습 중...
  [LGB] Fold val acc: 81.87%

===== Fold 3/5 =====
  [MLP] 학습 중...
  [MLP] Fold val acc: 82.20%
  [XGB] 학습 중...
  [XGB] Fold val acc: 82.80%
  [LGB] 학습 중...
  [LGB] Fold val acc: 82.10%

===== Fold 4/5 =====
  [MLP] 학습 중...
  [MLP] Fold val acc: 81.89%
  [XGB] 학습 중...
  [XGB] Fold val acc: 82.42%
  [LGB] 학습 중...
  [LGB] Fold val acc: 81.76%

===== Fold 5/5 =====
  [MLP] 학습 중...
  [MLP] Fold val acc: 82.94%
  [XGB] 학습 중...
  [XGB] Fold val acc: 83.06%
  [LGB] 학습 중...
  [LGB] Fold val acc: 82.38%

  K-Fold Cross Val